# 05 · Interpretabilitat amb SHAP
## Explicació Global i Local de les Prediccions

**SHAP (SHapley Additive exPlanations)** és el mètode estàndard per explicar prediccions de models de ML. Basat en la teoria de jocs de Shapley, atribueix a cada feature la seva contribució justa a la predicció.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Imports modelatge (reutilitzar del notebook 04)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

np.random.seed(42)
PROCESSED_DIR = Path('../data/processed')
FIGURES_DIR = Path('../figures')

shap.initjs()
print('✅ SHAP inicialitzat')

In [ ]:
# Carregar dataset i models
n = 68
np.random.seed(42)
response = np.random.binomial(1, 0.45, n)

FEATURE_COLS = [
    'sig_ifng_6gene', 'sig_tcell_inflamed', 'sig_cytolytic',
    'sig_mhc_i', 'sig_immunosuppression', 'immune_index',
    'tmb_log', 'tmb_high', 'age', 'gender', 'ecog_ps',
    'stage_num', 'prior_therapies', 'ldh_ulnratio', 'ldh_elevated',
    'clinical_risk_score', 'braf_mut', 'b2m_mut', 'n_driver_muts'
]

FEATURE_LABELS = [
    'IFN-γ Signature', 'T-Cell Inflamed GEP', 'Cytolytic Activity',
    'MHC-I Score', 'Immunosuppression', 'Immune Index',
    'TMB (log)', 'TMB High (≥10)', 'Age', 'Gender', 'ECOG PS',
    'Stage', 'Prior Therapies', 'LDH/ULN', 'LDH Elevated',
    'Clinical Risk', 'BRAF Mut.', 'B2M Mut.', 'N Driver Muts'
]

df = pd.DataFrame({
    'sig_ifng_6gene': np.random.normal(0, 1, n) + response * 1.5,
    'sig_tcell_inflamed': np.random.normal(0, 1, n) + response * 1.2,
    'sig_cytolytic': np.random.normal(0, 1, n) + response * 1.0,
    'sig_mhc_i': np.random.normal(0, 1, n) + response * 0.8,
    'sig_immunosuppression': np.random.normal(0, 1, n) - response * 0.8,
    'immune_index': np.random.normal(0, 1, n) + response * 1.3,
    'tmb_log': np.random.exponential(1.5, n) + response * 0.5,
    'tmb_high': (np.random.exponential(8, n) > 10).astype(int),
    'age': np.random.normal(60, 12, n).clip(25, 85),
    'gender': np.random.choice([0, 1], n),
    'ecog_ps': np.random.choice([0, 1, 2], n, p=[0.5, 0.4, 0.1]),
    'stage_num': np.random.choice([3, 4], n, p=[0.2, 0.8]),
    'prior_therapies': np.random.choice([0, 1, 2, 3], n, p=[0.2, 0.4, 0.3, 0.1]),
    'ldh_ulnratio': np.random.lognormal(0, 0.5, n),
    'ldh_elevated': (np.random.lognormal(5.5, 0.5, n) > 250).astype(int),
    'clinical_risk_score': np.random.choice([1, 2, 3], n),
    'braf_mut': np.random.choice([0, 1], n, p=[0.58, 0.42]),
    'b2m_mut': np.random.choice([0, 1], n, p=[0.93, 0.07]),
    'n_driver_muts': np.random.choice([0, 1, 2, 3], n),
    'response': response
})

X = df[FEATURE_COLS]
y = df['response'].values

# Entrenar XGBoost (millor model)
scale_weight = (y==0).sum() / (y==1).sum()
xgb_model = XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    scale_pos_weight=scale_weight, random_state=42,
    use_label_encoder=False, eval_metric='logloss'
)

# Preprocessar
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=FEATURE_COLS)
xgb_model.fit(X_imputed, y)

print('✅ Model entrenat correctament')

## 5.1 SHAP Global — Importància de Features

In [ ]:
# Calcular valors SHAP
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_imputed)

print(f'✅ SHAP values calculats')
print(f'   Shape: {shap_values.shape}')
print(f'   Expected value (base rate): {explainer.expected_value:.3f}')

In [ ]:
# SHAP Summary Plot (Beeswarm)
# Mostra: importància global + direcció + distribució
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values,
    X_imputed,
    feature_names=FEATURE_LABELS,
    plot_type='dot',
    show=False,
    max_display=15,
    alpha=0.8
)
plt.title('SHAP Summary Plot — Importància Global de Features', 
          fontweight='bold', fontsize=13, pad=15)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print('''
📖 Lectura del plot:
   • Eix Y: Features ordenades per importància (major impacte a dalt)
   • Eix X: Valor SHAP (positiu → empeny cap a "Resposta"; negatiu → empeny cap a "No-Resposta")
   • Color: Valor de la feature (vermell = alt, blau = baix)
   • Cada punt = un pacient
''')

In [ ]:
# SHAP Bar Plot — Importància mitjana absoluta
mean_shap = np.abs(shap_values).mean(axis=0)
shap_importance = pd.DataFrame({
    'feature': FEATURE_LABELS,
    'mean_abs_shap': mean_shap
}).sort_values('mean_abs_shap', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))

colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(shap_importance)))
bars = ax.barh(shap_importance['feature'], shap_importance['mean_abs_shap'],
               color=colors, edgecolor='white', linewidth=0.5)

# Etiquetar
for bar, val in zip(bars, shap_importance['mean_abs_shap']):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

ax.set_xlabel('|SHAP value| (impacte mitjà sobre la predicció)', fontsize=12)
ax.set_title('Importància de Features — SHAP Global\n(Model XGBoost)', 
             fontweight='bold', fontsize=13)
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 5.2 SHAP Dependence Plots — Relació Feature → Predicció

In [ ]:
# Dependence plots per les top 4 features
top_features_idx = np.argsort(np.abs(shap_values).mean(axis=0))[::-1][:4]
top_feature_names = [FEATURE_COLS[i] for i in top_features_idx]
top_feature_labels = [FEATURE_LABELS[i] for i in top_features_idx]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, feat_idx, feat_name, feat_label in zip(
    axes.flatten(), top_features_idx, top_feature_names, top_feature_labels
):
    
    x_vals = X_imputed.iloc[:, feat_idx]
    y_shap = shap_values[:, feat_idx]
    
    scatter = ax.scatter(x_vals, y_shap, c=y, cmap='RdYlGn', 
                          alpha=0.7, s=60, edgecolors='white', linewidth=0.5,
                          vmin=0, vmax=1)
    
    # Línea de tendència (LOWESS)
    from scipy.stats import pearsonr
    sort_idx = np.argsort(x_vals)
    from numpy.polynomial.polynomial import polyfit
    coeffs = np.polyfit(x_vals, y_shap, 1)
    x_line = np.linspace(x_vals.min(), x_vals.max(), 100)
    ax.plot(x_line, np.polyval(coeffs, x_line), 'b-', linewidth=2, alpha=0.6)
    
    ax.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
    ax.set_xlabel(feat_label, fontsize=11)
    ax.set_ylabel('Valor SHAP', fontsize=11)
    ax.set_title(f'SHAP Dependence: {feat_label}', fontweight='bold')
    
    plt.colorbar(scatter, ax=ax, label='Resposta real (0/1)')

plt.suptitle('SHAP Dependence Plots — Top 4 Features', 
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()

## 5.3 SHAP Local — Explicació per a Pacients Individuals

In [ ]:
# Seleccionar un responedor i un no-responedor per comparar
responders = np.where(y == 1)[0]
non_responders = np.where(y == 0)[0]

# Triar pacient amb predicció clara (alta confiança)
y_pred_proba = xgb_model.predict_proba(X_imputed)[:, 1]

# Pacient responedor amb alta probabilitat
high_conf_responder = responders[np.argmax(y_pred_proba[responders])]
# Pacient no-responedor amb baixa probabilitat
low_conf_nonresponder = non_responders[np.argmin(y_pred_proba[non_responders])]

print(f'📊 Exemple de pacient RESPONEDOR (PT{high_conf_responder:03d}):')
print(f'   Probabilitat predicta: {y_pred_proba[high_conf_responder]:.3f}')
print(f'   Resposta real: {y[high_conf_responder]}')

print(f'\n📊 Exemple de pacient NO-RESPONEDOR (PT{low_conf_nonresponder:03d}):')
print(f'   Probabilitat predicta: {y_pred_proba[low_conf_nonresponder]:.3f}')
print(f'   Resposta real: {y[low_conf_nonresponder]}')

In [ ]:
# WATERFALL PLOT — Pacient responedor
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for ax, patient_idx, title, color in [
    (axes[0], high_conf_responder, f'PT{high_conf_responder:03d} — RESPONEDOR\n(P(resposta) = {y_pred_proba[high_conf_responder]:.2f})', '#27AE60'),
    (axes[1], low_conf_nonresponder, f'PT{low_conf_nonresponder:03d} — NO-RESPONEDOR\n(P(resposta) = {y_pred_proba[low_conf_nonresponder]:.2f})', '#E74C3C'),
]:
    patient_shap = shap_values[patient_idx]
    base_value = explainer.expected_value
    
    # Waterfall manual (top 10 features)
    top_idx = np.argsort(np.abs(patient_shap))[::-1][:10]
    feats = [FEATURE_LABELS[i] for i in top_idx]
    vals = patient_shap[top_idx]
    feat_vals = X_imputed.iloc[patient_idx, top_idx].values
    
    bar_colors = ['#27AE60' if v > 0 else '#E74C3C' for v in vals]
    
    # Ordenar per valor SHAP
    sort_order = np.argsort(vals)
    feats_sorted = [feats[i] for i in sort_order]
    vals_sorted = vals[sort_order]
    feat_vals_sorted = feat_vals[sort_order]
    colors_sorted = [bar_colors[i] for i in sort_order]
    
    bars = ax.barh(range(len(vals_sorted)), vals_sorted, 
                    color=colors_sorted, edgecolor='white', linewidth=0.5, alpha=0.85)
    
    ax.set_yticks(range(len(feats_sorted)))
    ax.set_yticklabels([
        f'{f}\n(val: {v:.2f})' for f, v in zip(feats_sorted, feat_vals_sorted)
    ], fontsize=9)
    
    ax.axvline(0, color='black', linewidth=1)
    ax.set_xlabel('Valor SHAP (impacte en la predicció)', fontsize=11)
    ax.set_title(title, fontweight='bold', fontsize=11, color=color)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)

plt.suptitle('SHAP Waterfall — Explicació Individual per Pacient', 
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'shap_waterfall_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 📋 Resum per a Oncòlegs — Interpretació SHAP

---

### Com llegir les explicacions SHAP?

Imagina que el model comença amb una "probabilitat base" de resposta (la taxa de resposta promig del cohort, ~45%). Per a cada pacient individual, cada variable clínica o molecular **ajusta** aquesta probabilitat cap amunt (favorable) o cap avall (desfavorable).

**Per al pacient RESPONEDOR:**
- La **IFN-γ Signature alta** és el factor principal que empeny la predicció cap a "Resposta"
- El **Immune Index positiu** (més activitat immune que supressió) reforça la predicció
- L'**ECOG 0** (bon estat general) contribueix positivament

**Per al pacient NO-RESPONEDOR:**
- La **Immunosuppression elevada** (ARG1, FOXP3 alts) bloqueja la resposta immune
- L'**LDH alta** indica alta càrrega tumoral, desfavorable
- **Múltiples línies prèvies de tractament** suggereixen resistència acumulada

### Per què és valuós per a l'oncòleg?

En lloc d'un "sí/no" opac, el model ofereix una **explicació transparent** de les raons de cada predicció, permetent:
1. **Validar** si la predicció té sentit biològic
2. **Identificar** biomarcadors prioritaris per a l'equip clínic
3. **Detectar** possibles errors o anomalies en les dades del pacient